##### Step 1. Reloading data using Pandas pd.read_csv() method

In [2]:
import pandas as pd

df = pd.read_csv("~/Desktop/churn_risk_system/data/raw/European_Bank.csv")
df.head()

,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2025,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,2025,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,2025,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


##### Step 2. As we identified useless columns, by useless we mean that every column -

- **“Cannot help the model in any situation”**, we are going to drop them using Pandas df.drop() method.

In [3]:
df = df.drop(['CustomerId', 'Surname', 'Year'], axis=1)

##### Step 3. Separate features and target

In [4]:
features = df.drop('Exited', axis = 1)
target_Churn = df['Exited']

features

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,619,France,Female,42,2,0.00,1,1,1,101348.88
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58
2,502,France,Female,42,8,159660.80,3,1,0,113931.57
3,699,France,Female,39,1,0.00,2,0,0,93826.63
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10
...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77
9997,709,France,Female,36,7,0.00,1,0,1,42085.58
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52


In [5]:
target_Churn
# Left - Index, right - 1 churn/ 0 stayed in the output

0       1
1       0
2       1
3       0
4       0
       ..
9995    0
9996    0
9997    1
9998    1
9999    0
Name: Exited, Length: 10000, dtype: int64

##### Step 4. Encode categorical variables - By using **pd.get_dummies()** , for one-hot encoding, transforms non-numeric data (like strings or categories) into a numerical format usually binary 0-1, or Boolean True-False.

In [6]:
features = pd.get_dummies(features, drop_first =True )
features

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,False,False,False
1,608,41,1,83807.86,1,0,1,112542.58,False,True,False
2,502,42,8,159660.80,3,1,0,113931.57,False,False,False
3,699,39,1,0.00,2,0,0,93826.63,False,False,False
4,850,43,2,125510.82,1,1,1,79084.10,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,39,5,0.00,2,1,0,96270.64,False,False,True
9996,516,35,10,57369.61,1,1,1,101699.77,False,False,True
9997,709,36,7,0.00,1,0,1,42085.58,False,False,False
9998,772,42,3,75075.31,2,1,0,92888.52,True,False,True


##### Now that we got our categorical columns encoded. lets check the shape of our dataset.

In [7]:
features.shape

(10000, 11)

In [8]:
features.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,False,False,False
1,608,41,1,83807.86,1,0,1,112542.58,False,True,False
2,502,42,8,159660.80,3,1,0,113931.57,False,False,False
3,699,39,1,0.00,2,0,0,93826.63,False,False,False
4,850,43,2,125510.82,1,1,1,79084.10,False,True,False


In [9]:
features.columns

Index(['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Geography_Germany',
       'Geography_Spain', 'Gender_Male'],
      dtype='str')

#### **Summary of why we performed Data Encoding & Feature Preparation**

- Removed non-informative columns:
  - `CustomerId` (unique identifier)
  - `Surname` (irrelevant text)
  - `Year` (constant value)

---

- Separated the dataset into:
  - `features (X)` → input variables
  - `target_churn (y)` → output variable (`Exited`)

---

- Converted categorical variables into numerical format using one-hot encoding:
  - `Geography` → `Geography_Germany`, `Geography_Spain` (France as baseline)
  - `Gender` → `Gender_Male` (Female as baseline)

- Used `drop_first=True` to avoid redundancy and multicollinearity by keeping one category as a reference

- Final feature set includes:
  - Numerical: `CreditScore`, `Age`, `Tenure`, `Balance`, `NumOfProducts`, `EstimatedSalary`
  - Binary: `HasCrCard`, `IsActiveMember`
  - Encoded categorical: `Geography_Germany`, `Geography_Spain`, `Gender_Male`

- The dataset is now fully numeric and ready for model training

#### **Now we have Everything cleaned :**

clean ✅
correctly encoded ✅
ready for modeling


#### **So Far Done With**

Project structure ✔
EDA (strong insights) ✔
Feature understanding ✔
Data preprocessing ✔
Encoding ✔

---
---
### **Next step - Train-Test Split**

We split the dataset into training and testing sets (80/20), while keeping the churn distribution consistent, so the model can learn on one part and be fairly evaluated on unseen data.

    X_train - feature data used to train the model
    y_train - corresponding true values for X_train

    X_test - unseen feature data used for evaluation
    y_test - true values for X_test (used to compare against predictions)


In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    features, target_Churn, test_size= 0.2,
    stratify = target_Churn, random_state = 26)

In [11]:
X_train.shape

(8000, 11)

In [12]:
X_test.shape

(2000, 11)

In [14]:
y_train.value_counts(normalize=True)

Exited
0    0.79625
1    0.20375
Name: proportion, dtype: float64

In [27]:
y_test.value_counts(normalize=True)

Exited
0    0.7965
1    0.2035
Name: proportion, dtype: float64

As in the above step successfully split the data into 20-80% and cross checked, next Steps is Feature Scaling as different features have different ranges:

Like Age ranges from 20 to 80
Balance is up to 250,000

#### **Without scaling:**

large values dominates
and model becomes biased

In [28]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)


Imported scikitlearn a machine learning library, preprocessing its module for preparing data, StandardScaler to scale features. Then we created a scaler object so that we can use fit and transform methods in the next step.

- Fit learns from data. Fit method Calculates mean of each column from training, Calculates standard deviation from training
- Transform applies scaling formula
- We can also use combined method - fit_transform that does the both at once, just like above.

In [29]:
X_test_scaled = scaler.transform(X_test)

Scaling ensures that all features contribute equally to the model by bringing them to a comparable range.
- Age → -1 to +1
- Balance → -1 to +1
- Salary → -1 to +1

In [30]:
X_train_scaled[:5]

# checked the first 5 rows of scaled data

array([[-0.49044886,  0.30029821,  1.03379106,  0.24483435, -0.91299527,
        -1.55666057, -1.0307217 ,  0.76814155, -0.57658047,  1.7489578 ,
        -1.09499335],
       [ 1.68675428, -0.65541873, -0.00289408,  0.90869664,  0.7967337 ,
         0.6424008 , -1.0307217 , -0.19352961, -0.57658047, -0.57176909,
         0.91324755],
       [-0.7092109 ,  1.06487176, -1.03957921,  1.16326937, -0.91299527,
         0.6424008 ,  0.970194  ,  0.28580474, -0.57658047, -0.57176909,
         0.91324755],
       [-0.11542823, -0.46427534,  0.68822934,  1.17925788,  0.7967337 ,
        -1.55666057, -1.0307217 ,  0.97640179, -0.57658047,  1.7489578 ,
         0.91324755],
       [ 0.34293033,  0.3958699 , -0.34845579,  0.6337482 , -0.91299527,
         0.6424008 ,  0.970194  ,  0.29284716,  1.73436329, -0.57176909,
        -1.09499335]])

We use X_train_scaled[:5] to quickly inspect a small sample of the transformed data and verify that scaling worked correctly.

---
---

## SUMMARY : Data Preprocessing and Preparation

Following the exploratory data analysis, the dataset was prepared for machine learning modeling through a series of preprocessing steps. Non-informative features such as unique identifiers and irrelevant textual data were removed to ensure that only meaningful variables were retained.

The dataset was then divided into input features and the target variable representing customer churn. Categorical variables, including geography and gender, were transformed into numerical representations using one-hot encoding. To avoid redundancy and multicollinearity, one category from each encoded variable was dropped and treated as the baseline.

Subsequently, the dataset was split into training and testing sets using an 80:20 ratio. Stratified sampling was applied to maintain the original class distribution of churn across both sets, ensuring reliable model evaluation.

To further enhance model performance, feature scaling was performed using standardization. The scaling parameters were learned exclusively from the training data and then applied to the test data to prevent data leakage.

At the end of this phase, the dataset was fully prepared for modeling, with clean, scaled, and properly structured inputs suitable for training machine learning algorithms.

#### **Next Step Model Building!**